In [ ]:
import os
import cv2

DATA_DIR = './datase_add'
os.makedirs(DATA_DIR, exist_ok=True)

number_of_classes = 8
dataset_size = 300

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera!")
    exit()

stop_all = False  # press S

for j in range(number_of_classes):

    if stop_all:
        break

    class_dir = os.path.join(DATA_DIR, str(j))
    os.makedirs(class_dir, exist_ok=True)

    print(f'Collecting data class {j}')

    # wait
    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        cv2.putText(frame, ' s: start, q : stop', (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow('frame', frame)

        key = cv2.waitKey(25) & 0xFF
        if key == ord('s'):  # capt
            break
        if key == ord('q'):  # stop
            stop_all = True
            break

    if stop_all:
        break

    # capt
    counter = 0
    while counter < dataset_size:

        ret, frame = cap.read()
        if not ret:
            continue

        cv2.putText(frame, "", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
        cv2.imshow('frame', frame)

        key = cv2.waitKey(25) & 0xFF
        if key == ord('s'):  # stop everything
            stop_all = True
            break

        cv2.imwrite(os.path.join(class_dir, f'{counter}.jpg'), frame)
        counter += 1

    if stop_all:
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import os
import pickle

import mediapipe as mp
import cv2
import matplotlib.pyplot as plt

DATA_DIR = './datase_img'
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.3)


data = []
labels = []
for dir_ in os.listdir(DATA_DIR):
    for img_path in os.listdir(os.path.join(DATA_DIR, dir_)):
        data_aux = []

        x_ = []
        y_ = []

        img = cv2.imread(os.path.join(DATA_DIR, dir_, img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        results = hands.process(img_rgb)
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                for i in range(len(hand_landmarks.landmark)):
                    x = hand_landmarks.landmark[i].x
                    y = hand_landmarks.landmark[i].y

                    x_.append(x)
                    y_.append(y)

                for i in range(len(hand_landmarks.landmark)):
                    x = hand_landmarks.landmark[i].x
                    y = hand_landmarks.landmark[i].y
                    data_aux.append(x - min(x_))
                    data_aux.append(y - min(y_))

            data.append(data_aux)
            labels.append(dir_)

f = open('data.pickle', 'wb')
pickle.dump({'data': data, 'labels': labels}, f)
f.close()


In [ ]:
import pickle
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


data_dict = pickle.load(open('./data.pickle', 'rb'))

data = np.asarray(data_dict['data'])
labels = np.asarray(data_dict['labels'])

x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)

model = RandomForestClassifier()

model.fit(x_train, y_train)

y_predict = model.predict(x_test)

score = accuracy_score(y_predict, y_test)
accuracy  = accuracy_score(y_test, y_predict)
precision = precision_score(y_test, y_predict, average='weighted')
recall    = recall_score(y_test, y_predict, average='weighted')
f1        = f1_score(y_test, y_predict, average='weighted')


print('{}% of samples were classified correctly !'.format(score * 100))
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1-score : {f1 * 100:.2f}%")

print(classification_report(y_test, y_predict))


f = open('model.p', 'wb')
pickle.dump({'model': model}, f)
f.close()


99.62894248608535% of samples were classified correctly !
Accuracy : 99.63%
Precision: 99.63%
Recall   : 99.63%
F1-score : 99.63%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        80
           1       0.99      0.99      0.99        81
           2       1.00      1.00      1.00        60
           3       1.00      1.00      1.00        60
           4       0.98      0.98      0.98        60
           5       1.00      1.00      1.00        58
           6       1.00      1.00      1.00        60
           7       1.00      1.00      1.00        80

    accuracy                           1.00       539
   macro avg       1.00      1.00      1.00       539
weighted avg       1.00      1.00      1.00       539



In [ ]:
import pickle
import cv2
import mediapipe as mp
import numpy as np
import pyautogui
import time


MODEL_PATH = 'model.p' 
CONTROL_MAP = {
    0: 'Move Mouse',
    1: 'Stop Mouse',
    2: 'Left Click',
    3: 'Right Click',
    4: 'Scroll',
    5: 'Increase Volume (Thumbs Up)',
    6: 'Decrease Volume (Thumbs Down)',
    7: 'Screenshot'
}


try:
    with open(MODEL_PATH, 'rb') as f:
        model_dict = pickle.load(f)
    model = model_dict['model']
except FileNotFoundError:
    print(f"Error: Model file not found at {MODEL_PATH}. Please ensure '{MODEL_PATH}' exists.")
    exit()

# MediaPipe Hands 
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
hands = mp_hands.Hands(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)


cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: cant open ")
    exit()

#smooth mouse move
mouse_moving = False
SMOOTHNESS_FACTOR = 0.5 
last_action_time = time.time()
ACTION_COOLDOWN = 0.3 


def normalize_landmarks(hand_landmarks):
    data_aux = []
    x_ = [landmark.x for landmark in hand_landmarks.landmark]
    y_ = [landmark.y for landmark in hand_landmarks.landmark]

    min_x = min(x_)
    min_y = min(y_)

    #normalize coordinates 
    for i in range(len(hand_landmarks.landmark)):
        x = hand_landmarks.landmark[i].x
        y = hand_landmarks.landmark[i].y
        data_aux.append(x - min_x)
        data_aux.append(y - min_y)
    
    return np.asarray(data_aux)



def move_mouse(current_x, current_y, smooth=SMOOTHNESS_FACTOR):
    global mouse_moving
    
    # Map the hand coordinates 
    screen_width, screen_height = pyautogui.size()
    
    target_x = current_x * screen_width
    target_y = current_y * screen_height
    
    mouse_x, mouse_y = pyautogui.position()
    
    new_x = int(mouse_x + (target_x - mouse_x) * smooth)
    new_y = int(mouse_y + (target_y - mouse_y) * smooth)
    
    pyautogui.moveTo(new_x, new_y, _pause=False)
    mouse_moving = True

def perform_action(class_id):
    global last_action_time
    current_time = time.time()
    
    if current_time - last_action_time < ACTION_COOLDOWN:
        return
        
    action_name = CONTROL_MAP.get(class_id)

    if class_id == 2:  
        pyautogui.click()
    elif class_id == 3:  
        pyautogui.rightClick()
    elif class_id == 4: 
        pyautogui.scroll(-10) 
    elif class_id == 5: 
        pyautogui.press('volumeup')
    elif class_id == 6: 
        pyautogui.press('volumedown')
    elif class_id == 7: 
       
        try:
             pyautogui.screenshot(f'screenshot_{int(time.time())}.png')
        except Exception as e:
            print(f"Screenshot failed: {e}") 
        
    last_action_time = current_time
    print(f"Action Executed: {action_name}")


print("starting Hand gesture control - 'Q': quit.")


while True:
    ret, frame = cap.read()
    if not ret:
        break

    
    frame = cv2.flip(frame, 1)
    
    H, W, _ = frame.shape
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(frame_rgb)
    
    
    mouse_moving = False

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            
            # 1. Draw MediaPipe landmarks 
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

            # 2. Prepare data 
            normalized_data = normalize_landmarks(hand_landmarks)
            
            # Predict
            prediction = model.predict([normalized_data])[0]
            predicted_class_id = int(prediction)
            
            # Get action name
            predicted_action = CONTROL_MAP.get(predicted_class_id, "Unknown Action")
            
            # 3. Execute control action
            if predicted_class_id == 0:  # Move Mouse
               
                index_finger_tip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP]
                #
                move_mouse(index_finger_tip.x, index_finger_tip.y)
            elif predicted_class_id == 1: # Stop Mouse 
                pass
            else:
               
                perform_action(predicted_class_id)


            # Display predict
            
            # Calcule bounding box 
            x_coords = [landmark.x for landmark in hand_landmarks.landmark]
            y_coords = [landmark.y for landmark in hand_landmarks.landmark]

            x1 = int(min(x_coords) * W)
            y1 = int(min(y_coords) * H)
            x2 = int(max(x_coords) * W)
            y2 = int(max(y_coords) * H)
            
            # bounding box and predicted action 
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(frame, predicted_action, (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 2, cv2.LINE_AA)

    # Display overall status
    status_text = f"Status: {'MOVING' if mouse_moving else 'IDLE'}"
    cv2.putText(frame, status_text, (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)
    
    cv2.imshow('Hand Gesture Smart Control', frame)
    
    # stop 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()


ModuleNotFoundError: No module named 'mediapipe'